# Natural Language Processing + Decision Science

👩🏻‍🏫 In this challenge, we will combine:
* 🗣 Natural Language Processing
* 📊 Decision Science

🎯 The goal is to understand the **negative (bad) reviews** of products and sellers on Olist.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Data Manipulation
import numpy as np
import pandas as pd
pd.set_option("display.max_columns",None)

# Machine Learning
from sklearn.pipeline import make_pipeline

# Language Processing
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
import string
import unidecode as unidecode

# Vectorizers and NLP Models
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

🕵🏻‍♂️ Imagine that Olist CEO [Tiago Dalvi](https://www.linkedin.com/in/tiagodalvi/) asks you to read and understand the reviews.

- What did customers say when they rated their orders **1**, **2**, or **3** stars?
- What are the most common negative reviews?
    - About the worst-rated products?
    - About the worst-rated sellers?


## (0) Setup 🔨

First, we will load the DataFrame containing all information related to Olist reviews!

In [3]:
df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/olist_reviews.csv")

In [4]:
df.head()

,review_id,length_review,review_score,order_id,product_category_name,full_review
0,e64fb393e7b32834bb789ff8bb30750e,37,5,658677c97b385a9be170737859d3511b,ferramentas_jardim,Recebi bem antes do prazo estipulado.
1,f7c4243c7fe1938f181bec41a392bdeb,100,5,8e6bfb81e283fa7e4f11123a3fb894f1,esporte_lazer,Parabéns lojas lannister adorei comprar pela ...
2,8670d52e15e00043ae7de4c01cc2fe06,174,4,b9bf720beb4ab3728760088589c62129,eletroportateis,recomendo aparelho eficiente. no site a marca ...
3,4b49719c8a200003f700d3d986ea1a19,45,4,9d6f15f95d01e79bd1349cc208361f09,beleza_saude,"Mas um pouco ,travando...pelo valor ta Boa.\r\n"
4,3948b09f7c818e2d86c9a546758b2335,56,5,e51478e7e277a83743b6f9991dbfa3fb,informatica_acessorios,"Super recomendo Vendedor confiável, produto ok..."


In [5]:
df.shape

(36148, 6)

## (2) Text Cleaning

🧹 Create a `cleaning(sentence)` function and apply it to the reviews. **Note that there is no Portuguese lemmatizer in NLTK** (usually there is not, but there is the `nltk.stem.RSLPStemmer` stemmer).

In [7]:
nltk.download('rslp')

[nltk_data] Downloading package rslp to /Users/yaren/nltk_data...
[nltk_data]   Unzipping stemmers/rslp.zip.


True

In [8]:
def cleaning(sentence):
    sentence = sentence.strip()
    sentence = sentence.lower()
    sentence = "".join(char for char in sentence if not char.isdigit())

    for punctuation in string.punctuation:
        sentence = sentence.replace(punctuation, "") # remove punctuation
    
    tokenized_sentence = word_tokenize(sentence)
    stop_words = set(stopwords.words("portuguese")) # define stop words
    tokenized_sentence_cleaned = [word for word in tokenized_sentence if word not in stop_words] # remove stop words

    lemmatizer = nltk.stem.RSLPStemmer()
    tokenized_sentence_stemmed = [lemmatizer.stem(word) for word in tokenized_sentence_cleaned]

    return " ".join(tokenized_sentence_stemmed)

df["full_review_cleaned"] = df["full_review"].apply(cleaning)

In [9]:
df.head()

,review_id,length_review,review_score,order_id,product_category_name,full_review,full_review_cleaned
0,e64fb393e7b32834bb789ff8bb30750e,37,5,658677c97b385a9be170737859d3511b,ferramentas_jardim,Recebi bem antes do prazo estipulado.,receb bem ant praz estipul
1,f7c4243c7fe1938f181bec41a392bdeb,100,5,8e6bfb81e283fa7e4f11123a3fb894f1,esporte_lazer,Parabéns lojas lannister adorei comprar pela ...,parabém loj lannist ador compr internet segur ...
2,8670d52e15e00043ae7de4c01cc2fe06,174,4,b9bf720beb4ab3728760088589c62129,eletroportateis,recomendo aparelho eficiente. no site a marca ...,recom aparelh efici sit marc aparelh impress d...
3,4b49719c8a200003f700d3d986ea1a19,45,4,9d6f15f95d01e79bd1349cc208361f09,beleza_saude,"Mas um pouco ,travando...pelo valor ta Boa.\r\n",pouc travandopel val ta boa
4,3948b09f7c818e2d86c9a546758b2335,56,5,e51478e7e277a83743b6f9991dbfa3fb,informatica_acessorios,"Super recomendo Vendedor confiável, produto ok...",sup recom vend confi produt ok entreg ant praz


## (3) Analysis of Bad Reviews

### (3.1) Dataset with low review scores

😱 What proportion of reviews received a score between 1 and 3? 

In [13]:
bad_reviews_ratio = (df.review_score < 4).sum() / len(df)
print(f"{bad_reviews_ratio:.1%} of reviews have a score of 1, 2, or 3")

26.7% of reviews have a score of 1, 2, or 3


🕵🏻‍♂️ Let us focus on these reviews...

In [14]:
df[df.review_score < 4].head()

,review_id,length_review,review_score,order_id,product_category_name,full_review,full_review_cleaned
14,e233e51d11511bf30e568c76360ace52,97,1,548df2c6e5f089574614894bca78acf5,eletronicos,recebi somente 1 controle Midea Split ESTILO....,receb soment control mide split estil falt con...
23,8b230a1373c6dc4bd867099fda1d7039,60,3,071251fe3b3493294536f03737a8a679,ferramentas_jardim,Eu comprei duas unidades e só recebi uma e ag...,compr dua unidad receb agor faç
24,cb2fc3e5711b5ae85e0491ee18af63ed,72,3,34e6d418f368f8079ae152bc178bc66a,beleza_saude,"Produto bom, porém o que veio para mim não co...",produt bom porém vei mim condiz fot anúnci
25,60c714ed14cef913944a3147094a4742,36,1,9ac05114800f02bfaa783bd76842dbe2,moveis_decoracao,"Produto muito inferior, mal acabado.",produt inferi mal acab
28,0bd4dcc4f6c4621baf37f73495cad8c4,16,3,a11cd01ac67beef7e8bf09740d8a35c1,esporte_lazer,Entrega no prazo,entreg praz


### (3.2) Vectorization

🔡 ➡️ 🔢 Vectorize your texts.

- Make sure to account for **bigrams** (two-word expressions).
- Set `max_df = 0.75` to remove overly frequent words.
- Spoiler: you will end up with **20,000+** words…  
  For this challenge, limit to only `max_features = 5000`.

In [17]:
bad_reviews_df = df[df.review_score < 4]

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_df=0.75, max_features=5000)

vectorized_reviews = vectorizer.fit_transform(bad_reviews_df["full_review_cleaned"])
vectorized_reviews = pd.DataFrame(
    vectorized_reviews.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=bad_reviews_df.index
)

vectorized_reviews.head()

abaix  abert  aborrec  abr  abr caix  abr cham  abr embal  abr fech  \
14    0.0    0.0      0.0  0.0       0.0       0.0        0.0       0.0   
23    0.0    0.0      0.0  0.0       0.0       0.0        0.0       0.0   
24    0.0    0.0      0.0  0.0       0.0       0.0        0.0       0.0   
25    0.0    0.0      0.0  0.0       0.0       0.0        0.0       0.0   
28    0.0    0.0      0.0  0.0       0.0       0.0        0.0       0.0   

    abr reclam  abril  absurd  absurd entreg      acab  acab compr  acab deix  \
14         0.0    0.0     0.0            0.0  0.000000         0.0        0.0   
23         0.0    0.0     0.0            0.0  0.000000         0.0        0.0   
24         0.0    0.0     0.0            0.0  0.000000         0.0        0.0   
25         0.0    0.0     0.0            0.0  0.374942         0.0        0.0   
28         0.0    0.0     0.0            0.0  0.000000         0.0        0.0   

    acab fic  acab produt  acab receb  acab ruim  aceit  acend  acert  acess  \
14       0.0          0.0         0.0        0.0    0.0    0.0    0.0    0.0   
23       0.0          0.0         0.0        0.0    0.0    0.0    0.0    0.0   
24       0.0          0.0         0.0        0.0    0.0    0.0    0.0    0.0   
25       0.0          0.0         0.0        0.0    0.0    0.0    0.0    0.0   
28       0.0          0.0         0.0        0.0    0.0    0.0    0.0    0.0   

    acessóri  ach  ach absurd  ach bem  ach car  ach cord  ach demor  ach dev  \
14       0.0  0.0         0.0      0.0      0.0       0.0        0.0      0.0   
23       0.0  0.0         0.0      0.0      0.0       0.0        0.0      0.0   
24       0.0  0.0         0.0      0.0      0.0       0.0        0.0      0.0   
25       0.0  0.0         0.0      0.0      0.0       0.0        0.0      0.0   
28       0.0  0.0         0.0      0.0      0.0       0.0        0.0      0.0   

    ach entreg  ach estranh  ach fornec  ach fret  ach loj  ach mant  \
14         0.0          0.0         0.0       0.0      0.0       0.0   
23         0.0          0.0         0.0       0.0      0.0       0.0   
24         0.0          0.0         0.0       0.0      0.0       0.0   
25         0.0          0.0         0.0       0.0      0.0       0.0   
28         0.0          0.0         0.0       0.0      0.0       0.0   

    ach mater  ach melhor  ach pequen  ach pouc  ach praz  ach produt  \
14        0.0         0.0         0.0       0.0       0.0         0.0   
23        0.0         0.0         0.0       0.0       0.0         0.0   
24        0.0         0.0         0.0       0.0       0.0         0.0   
25        0.0         0.0         0.0       0.0       0.0         0.0   
28        0.0         0.0         0.0       0.0       0.0         0.0   

    ach péss  ach qual  ach vei  acid  acim  acion  acompanh  acompanh ped  \
14       0.0       0.0      0.0   0.0   0.0    0.0       0.0           0.0   
23       0.0       0.0      0.0   0.0   0.0    0.0       0.0           0.0   
24       0.0       0.0      0.0   0.0   0.0    0.0       0.0           0.0   
25       0.0       0.0      0.0   0.0   0.0    0.0       0.0           0.0   
28       0.0       0.0      0.0   0.0   0.0    0.0       0.0           0.0   

    acompanh produt  acondicion  acontec  acontec compr  acontec produt  \
14              0.0         0.0      0.0            0.0             0.0   
23              0.0         0.0      0.0            0.0             0.0   
24              0.0         0.0      0.0            0.0             0.0   
25              0.0         0.0      0.0            0.0             0.0   
28              0.0         0.0      0.0            0.0             0.0   

    acord  acord descr  acostum  acostum compr  acredit  acríl  acríl fic  \
14    0.0          0.0      0.0            0.0      0.0    0.0        0.0   
23    0.0          0.0      0.0            0.0      0.0    0.0        0.0   
24    0.0          0.0      0.0            0.0      0.0    0.0        0.0  

### (3.3) LDA

🕵🏻‍♂️ Fit LDA:
- Choose `n_components = 3`
- Show the Document-Topic Mixture with *.transform()*
- Show the Topic Mixture with *.components_*

In [18]:
n_components = 3
lda = LatentDirichletAllocation(n_components=n_components, random_state=42)
lda.fit(vectorized_reviews)

,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",3
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


#### Document Mixture (for Topics)

In [19]:
document_mixture = lda.transform(vectorized_reviews)
document_mixture

array([[0.07862797, 0.07843889, 0.84293314],
       [0.08294103, 0.08119877, 0.8358602 ],
       [0.08501857, 0.83717411, 0.07780732],
       ...,
       [0.15426997, 0.06646051, 0.77926951],
       [0.08864652, 0.09629065, 0.81506284],
       [0.81132625, 0.09709636, 0.0915774 ]])

👉 Let's report the most important topic for each review

In [23]:
most_important_topic = document_mixture.argmax(axis=1)
df.loc[bad_reviews_df.index, "most_important_topic"] = most_important_topic
most_important_topic

array([2, 2, 1, ..., 2, 2, 0])

#### Topic Mixture (for Words)

In [24]:
topic_word_mixture = lda.components_
topic_word_mixture

array([[ 1.4471101 ,  1.3027199 ,  2.68372512, ...,  0.33418771,
         1.25739753,  3.42847017],
       [ 3.22038338, 12.40199582,  0.36484821, ...,  1.42603701,
         2.06968493,  0.37297657],
       [ 0.34170859,  0.34474853,  0.35062439, ...,  0.33910428,
         0.43384833,  0.34205418]])

#### Topics

🎁 We provide some helper functions:
- `topic_word`: returns the most important words and their weights for a single topic
- `print_topics`: prints the different topics found by LDA with their most important words

In [25]:
def topic_word(vectorizer, model, topic, topwords, with_weights = True):
    topwords_indexes = topic.argsort()[:-topwords - 1:-1]
    if with_weights == True:
        topwords = [(vectorizer.get_feature_names_out()[i], round(topic[i],2)) for i in topwords_indexes]
    if with_weights == False:
        topwords = [vectorizer.get_feature_names_out()[i] for i in topwords_indexes]
    return topwords

In [26]:
def print_topics(vectorizer, model, topwords):
    for idx, topic in enumerate(model.components_):
        print("-"*20)
        print("Topic %d:" % (idx))
        print(topic_word(vectorizer, model, topic, topwords))


🕵🏻‍♂️ Print the topics with the most commonly used words:

In [34]:
print_topics(vectorizer, lda, 10)

--------------------
Topic 0:
[('produt', 201.36), ('receb', 186.07), ('vei', 139.23), ('receb produt', 133.17), ('gost', 106.32), ('nao', 106.06), ('compr', 105.45), ('err', 91.42), ('defeit', 87.82), ('difer', 75.59)]
--------------------
Topic 1:
[('bom', 230.0), ('produt', 227.43), ('entreg', 130.77), ('praz', 120.07), ('qual', 113.3), ('recom', 103.35), ('vei', 71.83), ('cheg', 70.56), ('bem', 66.11), ('péss', 60.98)]
--------------------
Topic 2:
[('entreg', 176.74), ('receb', 165.73), ('compr', 151.92), ('falt', 113.57), ('ped', 109.78), ('apen', 107.26), ('cheg', 106.1), ('produt', 93.56), ('doi', 89.12), ('aguard', 85.89)]


🇧🇷 Burada biraz Brezilya Portekizcesi kelimeler var:
- _cadeiras = chairs_
- _produto = product_
- _recomendo = recommend (não recomendo == not recommend)_
- _bom = good_
- _comprei = bought_
- _veio = came_
- _errado = wrong_
- _gostaria = I would like to..._
- _duas = two_
- _nao = not_
- _entregue = delivered_
- _pecas = part_
- _ainda = yet_
- _recebi = received_

👉 Show the most popular words associated with a topic

In [28]:
topic_word_mixture = [
    topic_word(vectorizer, lda, topic, topwords=10, with_weights=False)
    for topic in lda.components_
]

In [30]:
df["most_important_words"] = df["most_important_topic"].apply(
    lambda i: topic_word_mixture[int(i)] if pd.notna(i) else None
)

In [31]:
df[["review_id",
        "review_score",
        "product_category_name",
        "full_review_cleaned",
        "most_important_topic",
        "most_important_words"]
      ].head()

,review_id,review_score,product_category_name,full_review_cleaned,most_important_topic,most_important_words
0,e64fb393e7b32834bb789ff8bb30750e,5,ferramentas_jardim,receb bem ant praz estipul,NaN,None
1,f7c4243c7fe1938f181bec41a392bdeb,5,esporte_lazer,parabém loj lannist ador compr internet segur ...,NaN,None
2,8670d52e15e00043ae7de4c01cc2fe06,4,eletroportateis,recom aparelh efici sit marc aparelh impress d...,NaN,None
3,4b49719c8a200003f700d3d986ea1a19,4,beleza_saude,pouc travandopel val ta boa,NaN,None
4,3948b09f7c818e2d86c9a546758b2335,5,informatica_acessorios,sup recom vend confi produt ok entreg ant praz,NaN,None


In [35]:
df[df.review_score < 4][["review_id",
    "review_score",
    "product_category_name",
    "full_review_cleaned",
    "most_important_topic",
    "most_important_words"]
].head()

,review_id,review_score,product_category_name,full_review_cleaned,most_important_topic,most_important_words
14,e233e51d11511bf30e568c76360ace52,1,eletronicos,receb soment control mide split estil falt con...,2.0,"[entreg, receb, compr, falt, ped, apen, cheg, ..."
23,8b230a1373c6dc4bd867099fda1d7039,3,ferramentas_jardim,compr dua unidad receb agor faç,2.0,"[entreg, receb, compr, falt, ped, apen, cheg, ..."
24,cb2fc3e5711b5ae85e0491ee18af63ed,3,beleza_saude,produt bom porém vei mim condiz fot anúnci,1.0,"[bom, produt, entreg, praz, qual, recom, vei, ..."
25,60c714ed14cef913944a3147094a4742,1,moveis_decoracao,produt inferi mal acab,1.0,"[bom, produt, entreg, praz, qual, recom, vei, ..."
28,0bd4dcc4f6c4621baf37f73495cad8c4,3,esporte_lazer,entreg praz,1.0,"[bom, produt, entreg, praz, qual, recom, vei, ..."


## (3.4) Pipeline Tf-Idf and LDA

In [33]:
from sklearn import set_config
set_config("diagram")

🔨 Create a Pipeline connecting the previous Vectorizer and LDA.

Fit it on the cleaned texts.

In [39]:
pipeline = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), max_df=0.75, max_features=5000),
    LatentDirichletAllocation(n_components=3, random_state=42)
)
pipeline.fit(bad_reviews_df["full_review_cleaned"])

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidfvectorizer', ...), ('latentdirichletallocation', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


💡 If you try to access the components via `pipeline.components_`, it will NOT work because Pipeline does not have `components_`. However, you can use `pipeline._final_estimator` to access the LDA. And from there you can access the topics!

In [40]:
pipeline._final_estimator

,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",3
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


In [41]:
pipeline._final_estimator.components_

array([[ 1.4471101 ,  1.3027199 ,  2.68372512, ...,  0.33418771,
         1.25739753,  3.42847017],
       [ 3.22038338, 12.40199582,  0.36484821, ...,  1.42603701,
         2.06968493,  0.37297657],
       [ 0.34170859,  0.34474853,  0.35062439, ...,  0.33910428,
         0.43384833,  0.34205418]])

**Document Mixture** with Pipeline:

In [42]:
document_mixture = pipeline.transform(bad_reviews_df["full_review_cleaned"])
document_mixture

array([[0.07862797, 0.07843889, 0.84293314],
       [0.08294103, 0.08119877, 0.8358602 ],
       [0.08501857, 0.83717411, 0.07780732],
       ...,
       [0.15426997, 0.06646051, 0.77926951],
       [0.08864652, 0.09629065, 0.81506284],
       [0.81132625, 0.09709636, 0.0915774 ]])

**Topic Mixture** with Pipeline:

In [43]:
topic_mixture = pipeline._final_estimator.components_
topic_mixture

array([[ 1.4471101 ,  1.3027199 ,  2.68372512, ...,  0.33418771,
         1.25739753,  3.42847017],
       [ 3.22038338, 12.40199582,  0.36484821, ...,  1.42603701,
         2.06968493,  0.37297657],
       [ 0.34170859,  0.34474853,  0.35062439, ...,  0.33910428,
         0.43384833,  0.34205418]])

## (4) 🎁 Product Categories

### (4.1) Grouping by product categories

📈 Group the dataset by `product_category_name` and examine their performance.

In [44]:
# Product categories by performance - look at count, mean, median, and standard deviation
product_categories = df.groupby(by = 'product_category_name').agg({
        'review_score': ["count", "mean", "median", "std"]
    })

# Remove products with fewer sales than a threshold for analysis
cutoff = 50
product_categories = product_categories[product_categories[("review_score", "count")] > cutoff]

# Sort product categories by performance
product_categories = product_categories.sort_values(by = [('review_score', 'mean'),
                                                          ('review_score', 'std')],
                                                    ascending = [False, True])
product_categories

review_score                   \
                                                      count      mean median   
product_category_name                                                          
alimentos_bebidas                                        75  4.560000    5.0   
livros_interesse_geral                                  152  4.539474    5.0   
malas_acessorios                                        375  4.317333    5.0   
livros_tecnicos                                          83  4.301205    5.0   
pcs                                                      67  4.283582    5.0   
alimentos                                               151  4.231788    5.0   
papelaria                                               758  4.201847    5.0   
instrumentos_musicais                                   228  4.192982    5.0   
fashion_calcados                                        101  4.188119    5.0   
fashion_bolsas_e_acessorios                             726  4.168044    5.0   
perfumaria                                             1094  4.167276    5.0   
construcao_ferramentas_jardim                            85  4.164706    5.0   
cool_stuff                                             1360  4.154412    5.0   
ferramentas_jardim                                     1391  4.105679    5.0   
beleza_saude                                           3046  4.104071    5.0   
brinquedos                                             1271  4.101495    5.0   
esporte_lazer                                          2637  4.097838    5.0   
bebidas                                                 111  4.090090    5.0   
pet_shop                                                603  4.086235    5.0   
construcao_ferramentas_iluminacao                        86  4.081395    5.0   
eletroportateis                                         242  4.061983    5.0   
automotivo                                             1533  4.049576    5.0   
eletrodomesticos                                        354  4.033898    5.0   
eletrodomesticos_2                                       89  4.011236    5.0   
relogios_presentes                                     2211  4.009950    5.0   
construcao_ferramentas_construcao                       277  4.007220    5.0   
consoles_games                                          339  4.005900    5.0   
utilidades_domesticas                                  2192  4.005018    5.0   
eletronicos                                             942  3.979830    5.0   
industria_comercio_e_negocios                            82  3.963415    5.0   
bebes                                                   882  3.955782    5.0   
sinalizacao_e_seguranca                                  61  3.934426    5.0   
telefonia                                              1654  3.874244    5.0   
cama_mesa_banho                                        3891  3.853765    5.0   
artes                                                    80  3.850000    5.0   
casa_conforto                                           175  3.817143    5.0   
moveis_decoracao                                       2311  3.815232    5.0   
moveis_sala                                             138  3.789855    5.0   
informatica_acessorios                                 2473  3.774767    5.0   
audio                                                   134  3.768657    5.0   
market_place                                            102  3.745098    4.0   
moveis_cozinha_area_de_servico_jantar_e_jardim           94  3.734043    4.0   
construcao_ferramentas_seguranca                         61  3.721311    5.0   
agro_industria_e_comercio                                57  3.701754    4.0   
telefonia_fixa                                           96  3.697917    4.0   
casa_construcao                                         177  3.672316    4.0   
climatizacao                                             93  3.655914    4.0   
fashion_roupa_masculina                        

### (4.2) Worst product categories

👎 Save the five worst categories in terms of *average review score* into a variable called `worst_products`.

In [45]:
worst_products = product_categories.tail(5).sort_values(by = [("review_score", "count")],
                                                       ascending = False)
worst_products

review_score                           
                               count      mean median       std
product_category_name                                          
moveis_escritorio                546  3.298535    4.0  1.562366
casa_construcao                  177  3.672316    4.0  1.498094
telefonia_fixa                    96  3.697917    4.0  1.576938
climatizacao                      93  3.655914    4.0  1.625173
fashion_roupa_masculina           56  3.482143    4.0  1.737198

👇 Create a `worst_products_reviews` DataFrame containing only the items in `worst_products`.

In [46]:
worst_products_reviews = df[df.product_category_name.isin(worst_products.index)]
worst_products_reviews[["review_id",
                        "review_score",
                        "product_category_name",
                        "full_review_cleaned",
                        "most_important_topic",
                        "most_important_words"]
      ]

,review_id,review_score,product_category_name,full_review_cleaned,most_important_topic,most_important_words
45,c422274e50e900b46fee429016c5458d,3,moveis_escritorio,gost,0.0,"[produt, receb, vei, receb produt, gost, nao, ..."
126,5d6f9cddc8335878d8bf20c3bd4602e8,5,moveis_escritorio,meg recom receb produt corret dentr prazoverif...,NaN,None
212,1f836d160f50247a394034abd93b7c24,3,casa_construcao,esper instal,1.0,"[bom, produt, entreg, praz, qual, recom, vei, ..."
219,edd86e0af4dc62b340c60d846643cbf9,5,moveis_escritorio,produt ok praz estendidoporém entreg praz divu...,NaN,None
232,e47020cf52099bce9dc57524ea41218b,4,moveis_escritorio,indic loj compr sempr lannist,NaN,None
...,...,...,...,...,...,...
35956,123381026f13c723cb967b14b1bc673a,5,moveis_escritorio,ador cade bonit confort fácil mont mesm mont o...,NaN,None
36031,042b7c995635364a03c5bb79e994f513,4,fashion_roupa_masculina,filh gost,NaN,None
36034,5d4beea8e2c71baee7f55c7ae3ae3313,1,moveis_escritorio,insatisfeit cade vei defeit fabr solt encaix p...,0.0,"[produt, receb, vei, receb produt, gost, nao, ..."
36037,0cac8fef1bcba2a3335b132807520f6d,1,moveis_escritorio,produt vei fur err mont infeliz cade vai dur m...,1.0,"[bom, produt, entreg, praz, qual, recom, vei, ..."


### (4.3) Topics for the worst products

❓ What are the topics of the worst products? ❓

In [47]:
worst_products_reviews["most_important_topic"].value_counts()

most_important_topic
2.0    160
1.0    148
0.0    106
Name: count, dtype: int64

In [48]:
bad_frequency = list(worst_products_reviews["most_important_topic"].value_counts().index)
bad_frequency

[2.0, 1.0, 0.0]

In [51]:
[topic_word_mixture[int(i)] for i in bad_frequency]

[['entreg',
  'receb',
  'compr',
  'falt',
  'ped',
  'apen',
  'cheg',
  'produt',
  'doi',
  'aguard'],
 ['bom',
  'produt',
  'entreg',
  'praz',
  'qual',
  'recom',
  'vei',
  'cheg',
  'bem',
  'péss'],
 ['produt',
  'receb',
  'vei',
  'receb produt',
  'gost',
  'nao',
  'compr',
  'err',
  'defeit',
  'difer']]

## (5) 🎁 Sellers...

* What kind of products were sold by the worst sellers?
* What are the main reviews for the worst sellers?

### (5.1) Worst sellers

In [ ]:
from olist.seller import Seller
sellers = Seller().get_training_data()
sellers.columns

👇 Select the 10 worst-selling sellers and save them into a variable called `worst_sellers`.

In [ ]:
worst_sellers = sellers[["seller_id", "review_score", "profits"]].sort_values(
    by = "profits",
    ascending = True).head(10)
worst_sellers

### (5.2) Products sold by the worst sellers

In [ ]:
products = Product().get_training_data() [["product_id", "category"]]
products

❓ What types of products are sold by the worst sellers? ❓

In [ ]:
sellers_product_category = data["order_items"].merge(products,
                                             on = "product_id", how = "left")[["seller_id", "category"]]

sellers_product_category

In [ ]:
sellers_product_category.groupby("seller_id").count()

### (5.3) Categories and topics for the worst sellers

🎁 Here are some useful functions:
- `focus_seller(seller_id)` to show product categories sold by a seller
- `bad_reviews_seller` to show the most popular words of the most frequent topics for a seller

In [ ]:
def focus_seller(seller_id):
    return sellers_product_category[sellers_product_category.seller_id == seller_id].value_counts()

In [ ]:
bad_reviews_sellers = worst_products_reviews.merge(data["order_items"])
bad_reviews_sellers.head(3)

In [ ]:
def bad_reviews_seller(bad_reviews_sellers, seller_id):
    mask = (bad_reviews_sellers.seller_id == seller_id)
    temp = bad_reviews_sellers[mask]
    if len(temp) > 0: # if seller appears in the bad reviews dataframe
        most_frequent_topic_seller = list(temp.most_important_topic.value_counts().head(1).index)[0]
        return topic_word_mixture[most_frequent_topic_seller]

❓ For each of these worst-selling sellers, show the most frequent product categories and words ❓

In [ ]:
for worst_seller in worst_sellers["seller_id"]:
    print("-"*50)
    print(f"Focusing on the seller #{worst_seller}...")
    print(focus_seller(worst_seller))
    print(bad_reviews_seller(bad_reviews_sellers, worst_seller))


🏁 Congratulations. You have learned some NLP basics (Preprocessing + Vectorization + NB/LDA) and combined this new "expertise" with Decision Science.

💾 Do not forget to `git add / commit / push`.